# Practical 3: Data Cleaning and Preprocessing for Downstream Analysis
**Course:** Python for Data Science (`BE05000231_PDS`)  
**Semester:** 5th Semester B.E. / B.Tech  
**Topic:** Datetime Harmonization, Missing Value Imputation, RFC 3986 URL Normalization, and Case/Whitespace Standardization  
**Primary Pipeline Input:** Practical 2 Structured Dataset (`structured_logs.csv` — 300,010 records)  
**External Real-World Benchmark:** Elastic Official Production Web Log (`elastic_structured_logs.csv` — 10,000 records)  

---

## 1. AIM & OBJECTIVES
### Aim
To clean, impute, and preprocess structured web server access log data for downstream feature engineering and threat classification, enforcing temporal validity, missing value resolution, canonical URL path normalization, and whitespace harmonization.

### Objectives
1. **Datetime Conversion:** Parse raw NCSA timestamp strings into standardized timezone-aware ISO-8601 UTC datetime objects (`datetime_utc`), deriving hour and day-of-week attributes.
2. **Missing Value Imputation:** Resolve null/empty entries in `ident`, `auth_user`, `referrer`, `user_agent`, and `payload` using domain-grounded cybersecurity defaults.
3. **Data Consistency & Bound Enforcement:** Validate numeric boundaries (`bytes_sent >= 0`, `status_code` in `[100, 599]`) and rectify anomalies.
4. **RFC 3986 URL Path Normalization:** Implement URL unquoting, lowercase paths, collapse redundant slashes (`//` $\rightarrow$ `/`), normalize default index files (`/index.html` $\rightarrow$ `/`), and isolate clean query payloads.
5. **Whitespace & Casing Standardization:** Strip leading/trailing spaces and collapse repeated internal whitespaces in headers and URIs.
6. **Pipeline Continuity & Export:** Ingest Practical 2 structured tables and export clean tables in CSV and Snappy Parquet formats.
7. **External Benchmark:** Validate the end-to-end cleaning workflow on Elastic's real-world production server telemetry.

---
## 2. THEORETICAL FOUNDATION: DATA CLEANING IN LOG INTELLIGENCE

### 2.1 The Need for Normalization
In raw server logs, clients interact through heterogeneous user agents, proxies, and encoders. Without data cleaning:
- An attack payload at `/STORE/ITEM?ID=1` and `/store//item/?id=1` would be treated as two completely distinct resources by machine learning models.
- Missing headers (`-`) would cause `NaN` propagation, crashing numerical classifiers.
- String timestamps cannot be queried using temporal windowing or rolling frequency deltas (critical for DDoS detection).

### 2.2 Canonical Cleaning Rules
| Field | Raw State | Cleaning Action | Normalized State |
|---|---|---|---|
| `timestamp` | `08/Oct/2026:08:15:32 +0000` | Parse `%d/%b/%Y:%H:%M:%S %z`, forward-fill | `2026-10-08 08:15:32+00:00` (UTC) |
| `ident`, `auth_user` | `NaN`, `""` | RFC 1413 / Basic Auth absent | `"-"` |
| `referrer` | `NaN`, `""` | Direct navigation or scanner | `"-"` |
| `user_agent` | `NaN`, `""` | Unspecified client | `"unknown"` |
| `resource_requested` | `//STORE/index.html?id=1` | Unquote, lowercase path, strip index | Path: `/store`, URL: `/store?id=1` |
| `bytes_sent` | `< 0`, `NaN` | Non-negative integer boundary | `0` (clipped) |

---
## 3. IMPORT LIBRARIES & CONFIGURE PATHS
We import `pandas`, `numpy`, `re`, `urllib.parse`, and `matplotlib`.

In [ ]:
import sys
import os
import time
import re
import urllib.parse
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Resolve directories across notebook or CLI context
PRACTICAL_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
WORKSPACE_DIR = PRACTICAL_DIR.parent

PRACTICAL2_DATA = WORKSPACE_DIR / 'Practical 2' / 'data' / 'processed'
PROCESSED_DIR = PRACTICAL_DIR / 'data' / 'processed'
PLOTS_DIR = PRACTICAL_DIR / 'outputs' / 'plots'
REPORTS_DIR = PRACTICAL_DIR / 'outputs' / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_INPUT = PRACTICAL2_DATA / 'structured_logs.csv'
EXTERNAL_INPUT = PRACTICAL2_DATA / 'elastic_structured_logs.csv'

print(f"Primary Input File   : {PRIMARY_INPUT} (Exists: {PRIMARY_INPUT.exists()})")
print(f"External Benchmark   : {EXTERNAL_INPUT} (Exists: {EXTERNAL_INPUT.exists()})")


---
## 4. INGEST PRACTICAL 2 STRUCTURED TELEMETRY
We load the structured DataFrame generated by Practical 2 and inspect baseline nullity.

In [ ]:
t0 = time.time()
df = pd.read_csv(PRIMARY_INPUT)
print(f"Loaded Primary Dataset: {len(df):,} rows x {len(df.columns)} columns in {time.time()-t0:.2f}s")
print("\n--- Baseline Null Values Before Cleaning ---")
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head(3)


---
## 5. RFC 3986 URL PATH NORMALIZATION ENGINE
We define our cached, high-throughput URL normalizer to harmonize paths and collapse directory redundancy.

In [ ]:
_CACHE = {}
INDEX_REGEX = re.compile(r"/(index|default)\.(html?|php|asp|jsp)$", re.IGNORECASE)
SLASH_REGEX = re.compile(r"/+")
SPACE_REGEX = re.compile(r"\s+")

def normalize_uri(uri: str):
    if not isinstance(uri, str) or not uri or uri == '-':
        return '/', '', '/'
    if uri in _CACHE:
        return _CACHE[uri]
    u = uri.strip()
    if '?' in u:
        path_part, query_part = u.split('?', 1)
    else:
        path_part, query_part = u, ''
    
    p_clean = urllib.parse.unquote(path_part).strip().lower()
    p_clean = SLASH_REGEX.sub('/', p_clean)
    p_clean = INDEX_REGEX.sub('', p_clean)
    if len(p_clean) > 1 and p_clean.endswith('/'):
        p_clean = p_clean[:-1]
    if not p_clean:
        p_clean = '/'
    q_clean = SPACE_REGEX.sub(' ', query_part.strip())
    norm = f"{p_clean}?{q_clean}" if q_clean else p_clean
    res = (p_clean, q_clean, norm)
    _CACHE[uri] = res
    return res

# Demonstration on typical test vectors
test_samples = [
    "/index.html",
    "//store///item/?id=45",
    "/ADMIN/LOGIN.PHP",
    "/search?q=%3Cscript%3Ealert(1)%3C/script%3E",
    "/store/item?RotomMow   Rotom"
]
print("URL Normalization Demonstrations:")
for s in test_samples:
    p, q, n = normalize_uri(s)
    print(f"  * Raw: {s:<42} -> Clean Path: {p:<15} | Norm: {n}")


---
## 6. EXECUTE FULL PREPROCESSING PIPELINE
We execute the full cleaning pipeline on the primary dataset:
1. Forward-fill sequential timestamps and convert to UTC ISO datetime.
2. Impute missing headers (`ident`, `auth_user`, `referrer`, `user_agent`, `payload`).
3. Enforce valid numeric bounds on `bytes_sent` and `status_code`.
4. Clean and normalize all 300,010 URI paths.
5. Flag rapid duplicate queries.

In [ ]:
t_clean = time.time()

# 1. Datetime Conversion
df['timestamp'] = df['timestamp'].ffill().fillna('01/Jan/1970:00:00:00 +0000')
df['datetime_utc'] = pd.to_datetime(df['timestamp'], format='%d/%b/%Y:%H:%M:%S %z', errors='coerce')
df['hour'] = df['datetime_utc'].dt.hour.fillna(0).astype(np.int8)
df['day_of_week'] = df['datetime_utc'].dt.dayofweek.fillna(0).astype(np.int8)

# 2. Missing Value Imputation
df['ident'] = df['ident'].fillna('-').replace('', '-')
df['auth_user'] = df['auth_user'].fillna('-').replace('', '-')
df['referrer'] = df['referrer'].fillna('-').replace('', '-').str.strip()
df['user_agent'] = df['user_agent'].fillna('unknown').replace('', 'unknown').str.strip()
df['payload'] = df['payload'].fillna('').astype(str)

# 3. Bounds enforcement
df['bytes_sent'] = pd.to_numeric(df['bytes_sent'], errors='coerce').fillna(0).clip(lower=0).astype(np.int32)
df['status_code'] = pd.to_numeric(df['status_code'], errors='coerce').fillna(200).astype(np.int16)

# 4. String cleaning & URL Normalization
df['client_ip'] = df['client_ip'].astype(str).str.strip()
df['request_method'] = df['request_method'].astype(str).str.strip().str.upper()
df['http_protocol'] = df['http_protocol'].astype(str).str.strip().str.upper()

norm_res = [normalize_uri(u) for u in df['resource_requested'].tolist()]
df['clean_path'] = [r[0] for r in norm_res]
df['clean_payload'] = [r[1] for r in norm_res]
df['normalized_url'] = [r[2] for r in norm_res]

# 5. Rapid duplicate check
dup_mask = df.duplicated(subset=['client_ip', 'datetime_utc', 'request_method', 'normalized_url', 'status_code'], keep='first')
df['is_rapid_duplicate'] = dup_mask.astype(np.int8)

elapsed = time.time() - t_clean
print(f"Cleaning Complete in {elapsed:.2f}s ({len(df)/elapsed:,.0f} rows/sec)!")
print(f"Nulls Remaining: {df.isnull().sum().sum()} (100% Complete!)")
print(f"Rapid Consecutive Duplicates: {int(dup_mask.sum()):,}")
df[['client_ip', 'datetime_utc', 'request_method', 'clean_path', 'status_code', 'is_rapid_duplicate']].head(5)


---
## 7. EXTERNAL BENCHMARK: ELASTIC PRODUCTION TELEMETRY
We benchmark the cleaning workflow against Elastic's official production web access log (10,000 records).

In [ ]:
df_ext = pd.read_csv(EXTERNAL_INPUT)
t_ext = time.time()

# Apply identical cleaning steps
df_ext['timestamp'] = df_ext['timestamp'].ffill().fillna('01/Jan/1970:00:00:00 +0000')
df_ext['datetime_utc'] = pd.to_datetime(df_ext['timestamp'], format='%d/%b/%Y:%H:%M:%S %z', errors='coerce')
df_ext['hour'] = df_ext['datetime_utc'].dt.hour.fillna(0).astype(np.int8)
df_ext['day_of_week'] = df_ext['datetime_utc'].dt.dayofweek.fillna(0).astype(np.int8)
df_ext['ident'] = df_ext['ident'].fillna('-').replace('', '-')
df_ext['auth_user'] = df_ext['auth_user'].fillna('-').replace('', '-')
df_ext['referrer'] = df_ext['referrer'].fillna('-').replace('', '-').str.strip()
df_ext['user_agent'] = df_ext['user_agent'].fillna('unknown').replace('', 'unknown').str.strip()
df_ext['payload'] = df_ext['payload'].fillna('').astype(str)
df_ext['bytes_sent'] = pd.to_numeric(df_ext['bytes_sent'], errors='coerce').fillna(0).clip(lower=0).astype(np.int32)
df_ext['status_code'] = pd.to_numeric(df_ext['status_code'], errors='coerce').fillna(200).astype(np.int16)

ext_norm = [normalize_uri(u) for u in df_ext['resource_requested'].tolist()]
df_ext['clean_path'] = [r[0] for r in ext_norm]
df_ext['clean_payload'] = [r[1] for r in ext_norm]
df_ext['normalized_url'] = [r[2] for r in ext_norm]

ext_dup = df_ext.duplicated(subset=['client_ip', 'datetime_utc', 'request_method', 'normalized_url', 'status_code'], keep='first')
df_ext['is_rapid_duplicate'] = ext_dup.astype(np.int8)

print("=" * 65)
print(f"External Elastic Dataset  : {len(df_ext):,} records cleaned in {time.time()-t_ext:.3f}s")
print(f"Unique Clean Paths        : {df_ext['clean_path'].nunique():,}")
print(f"Nulls Remaining           : {df_ext.isnull().sum().sum()}")
print(f"Rapid Consecutive Dups    : {int(ext_dup.sum()):,}")
print("=" * 65)
df_ext[['client_ip', 'datetime_utc', 'clean_path', 'status_code']].head(3)


---
## 8. SERIALIZE PREPROCESSED DATASETS
We export the clean tabular records to both CSV and Snappy-compressed Apache Parquet.

In [ ]:
csv_out = PROCESSED_DIR / 'preprocessed_logs.csv'
parquet_out = PROCESSED_DIR / 'preprocessed_logs.parquet'
ext_out = PROCESSED_DIR / 'elastic_preprocessed_logs.csv'

df.to_csv(csv_out, index=False)
df.to_parquet(parquet_out, index=False, compression='snappy')
df_ext.to_csv(ext_out, index=False)

print(f"Exported Primary CSV     : {csv_out.name} ({csv_out.stat().st_size / (1024*1024):.2f} MB)")
print(f"Exported Primary Parquet : {parquet_out.name} ({parquet_out.stat().st_size / (1024*1024):.2f} MB)")
print(f"Exported External CSV    : {ext_out.name} ({ext_out.stat().st_size / (1024*1024):.2f} MB)")


---
## 9. EXPLORATORY DIAGNOSTIC VISUALIZATIONS
We generate publication-grade visual charts demonstrating the data cleaning impact.

In [ ]:
# 1. Top Clean URI Paths
plt.figure(figsize=(9, 4.5), dpi=150)
top_p = df['clean_path'].value_counts().head(8)
colors = ['#1687FF', '#22D3EE', '#21D98B', '#FFB547', '#9333EA', '#EC4899', '#3B82F6', '#10B981']
plt.barh(top_p.index.astype(str), top_p.values, color=colors[:len(top_p)], edgecolor='#060912')
plt.title('Top Normalized Clean URI Paths (RFC 3986)', fontsize=12, fontweight='bold')
plt.xlabel('Request Frequency', fontsize=10)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# 2. Hourly Traffic Breakdown
plt.figure(figsize=(10, 4), dpi=150)
hourly = df['datetime_utc'].dt.floor('h').value_counts().sort_index()
plt.plot(hourly.index, hourly.values, color='#1687FF', linewidth=2, marker='o', markersize=3)
plt.fill_between(hourly.index, hourly.values, color='#1687FF', alpha=0.15)
plt.title('Timeline Request Volume (Standardized UTC Datetime)', fontsize=12, fontweight='bold')
plt.xlabel('Timestamp (UTC)', fontsize=10)
plt.ylabel('Requests / Hour', fontsize=10)
plt.tight_layout()
plt.show()
